## PATSTAT persons - lesson 3
This notebook expands on the second lesson about PATSTAT. We take a look at the two tables that contain information about inventors and applicants, and learn to join tables using ORM.

## The tls207_pers_appln persons table

PATSTAT links two types of persons to a given application: inventors and applicants. Please note that applicants can be phisical persons or legal entities such as corporations. There can be a many-to-many relation between persons and applications. The `tls207_pers_appln` table serves as an intermediary table that links applications from the `tls201_appln` table to persons (both inventors and applicants). 

#### Fields in tls207_pers_appln
  
- `appln_id`: this field is a foreign key that references the `appln_id` in the `tls201_appln` table, establishing a link between an application and its related persons.
- `person_id`: this field is a foreign key that references the `person_id` in the `tls206_person` table, identifying the person associated with the application.
- `applt_seq_nr`: this field denotes the sequence number for applicants, allowing the identification of multiple applicants for a single application.
- `invt_seq_nr`: this field denotes the sequence number for inventors, distinguishing inventors from other types of persons associated with the application.

## The most influential inventor of the decade
In lesson 2 we used the `nb_citing_docdb_fam` field of the applications table to determine the granted European patents that have been cited by other patents the most, as a proxy for finding the most influential invention of the decade. 

In this lesson we will find out who is the most influential inventor by the same metric. First we need to know who are the inventors listed in each application from our query. 

### Getting the persons per application
To find the persons associated with an application, we query PATSTAT with a `JOIN` between `tls207_pers_appln` and `tls201_appln`. We use the appln_id for the join, as this is the field shared between the two tables. 

In [1]:
# Importing the PATSTAT client
from epo.tipdata.patstat import PatstatClient

# Initialise the PATSTAT client
patstat = PatstatClient()

# Access ORM
db = patstat.orm()

# Importing tables as models
from epo.tipdata.patstat.database.models import TLS201_APPLN, TLS207_PERS_APPLN

This client instance is currently configured to use a test dataset with reduced number of publications (~10K).
Use PatstatClient(env='PROD') to use the complete PATSTAT dataset (>140M publications).
Use PatstatClient(env='TEST') to use the test dataset and avoid displaying this warning



In [2]:
# Define the query in ORM
q = db.query(
    TLS201_APPLN.appln_id,
    TLS201_APPLN.appln_nr,
    TLS201_APPLN.nb_citing_docdb_fam,  
    TLS207_PERS_APPLN.person_id,     # person ID from table 207
    TLS207_PERS_APPLN.invt_seq_nr,   # inventor sequence number from table 207, different than 0 if the person is an inventor
    TLS207_PERS_APPLN.applt_seq_nr   # applicant sequence number from table 207, different than 0 if the person is an applicant
).join(
    TLS207_PERS_APPLN, TLS201_APPLN.appln_id == TLS207_PERS_APPLN.appln_id # tables 201 and 207 are joined with the common field appln_id
).filter(
    TLS201_APPLN.appln_filing_year >= 2020,
    TLS201_APPLN.appln_auth == 'EP',
    TLS201_APPLN.granted == 'Y'
).order_by(
    TLS201_APPLN.appln_id.desc()
)

# Creating a dataframe with the results
res = patstat.df(q)

res


,appln_id,appln_nr,nb_citing_docdb_fam,person_id,invt_seq_nr,applt_seq_nr
0,589555726,23164131,1,47829593,0,1
1,589555726,23164131,1,83215891,1,0
2,578613921,22192729,0,88864021,2,0
3,578613921,22192729,0,90553894,0,1
4,578613921,22192729,0,88919408,1,0
...,...,...,...,...,...,...
1958,524556586,20150555,1,74584733,1,0
1959,524556586,20150555,1,5024,0,1
1960,524398757,20150262,4,74502076,2,0
1961,524398757,20150262,4,74542589,1,0


#### Understanding the results 
Since we joined two tables, we can see that we can have multiple entries for a given application ID, while when we query the applications table the application the appliction ID is unique. This happens because an application will be linked with at least two persons, one applicant and one inventor. We can also see that some persons have a 0 in the `invt_seq_nr` field, and a value greater than 0 in the `applt_seq_nr`. This means that those persons are just applicants. It is important to note that in some applications the inventor and the applicant may be the same person.

### Filtering out applicants
 We need to filter out the persons who have a value 0 in the inventor sequence number, since those persons are **just** applicants. 

In [3]:
# Importing necessary modules
from epo.tipdata.patstat.database.models import TLS201_APPLN, TLS207_PERS_APPLN

# Define the query in ORM
q = db.query(
    TLS201_APPLN.appln_id,
    TLS201_APPLN.appln_nr,
    TLS201_APPLN.nb_citing_docdb_fam,  # number of families citing the application
    TLS207_PERS_APPLN.person_id
).join(
    TLS207_PERS_APPLN, TLS201_APPLN.appln_id == TLS207_PERS_APPLN.appln_id
).filter(
    TLS201_APPLN.appln_filing_year >= 2020,
    TLS201_APPLN.appln_auth == 'EP',
    TLS201_APPLN.granted == 'Y',
    TLS207_PERS_APPLN.invt_seq_nr > 0  # filter to exclude applicants that are not inventors
).order_by(
    TLS207_PERS_APPLN.person_id.desc()
)

# Creating a dataframe with the results
res = patstat.df(q)

res


,appln_id,appln_nr,nb_citing_docdb_fam,person_id
0,556052791,21192581,0,90692578
1,540640526,20206880,0,90683805
2,529019668,20715065,10,90669898
3,533161693,20181061,8,90657133
4,536945187,20760107,0,90655366
...,...,...,...,...
1412,537592943,20765210,0,298210
1413,540375137,20206131,0,274311
1414,538660423,20776141,1,72854
1415,532823928,20730571,0,1221


### The tls206_person table
We have the `person_id` in the table above as the unique identifier of the inventors with granted patents that have been filed this decade, but we do not know their names. We need another table for that. 

The `tls206_person` table contains details about inventors and applicants.This table is typically joined with the `tls207_pers_appln` table to link persons to specific patent applications.


#### Fields of tls206_person:
- **person_id**: unique identifier for each person.
- **person_name**: name of the person or organisation.
- **doc_std_name**: standardised name for the person or organisation.
- **person_address**: address of the person.
- **person_ctry_code**: country code associated with the person.
- **psn_sector**: sector classification of the person (e.g., academia, industry).


#### Adding the person name to the query
We need to add a further `JOIN` to our query, where we join the `TLS207_PERS_APPLN` and the `TLS206_PERSON` tables, with the common field `person_id`. We will now have three tables joined as a result. 

In [4]:
# Importing table 206
from epo.tipdata.patstat.database.models import TLS201_APPLN, TLS207_PERS_APPLN, TLS206_PERSON

# Define the query in ORM
q = db.query(
    TLS201_APPLN.appln_id,
    TLS201_APPLN.appln_nr,
    TLS201_APPLN.appln_filing_date,
    TLS201_APPLN.nb_citing_docdb_fam,
    TLS207_PERS_APPLN.person_id,
    TLS206_PERSON.person_name  # inventor's name as found in table 206
).join(
    TLS207_PERS_APPLN, TLS201_APPLN.appln_id == TLS207_PERS_APPLN.appln_id  # tables 201 and 207 are joined with the common field appln_id
).join(
    TLS206_PERSON, TLS207_PERS_APPLN.person_id == TLS206_PERSON.person_id   # tables 206 and 207 are joined with the common field person_id
).filter(
    TLS201_APPLN.appln_filing_year >= 2020,
    TLS201_APPLN.appln_auth == 'EP',
    TLS201_APPLN.granted == 'Y',
    TLS207_PERS_APPLN.invt_seq_nr > 0  
).order_by(
    TLS206_PERSON.person_name.desc()
)

# Creating a dataframe with the results
res = patstat.df(q)

res


,appln_id,appln_nr,appln_filing_date,nb_citing_docdb_fam,person_id,person_name
0,540640526,20206880,2020-11-11,0,79407945,"van de Pol, Aart"
1,543055501,20000457,2020-12-12,5,79423861,"ten Thije, Niels"
2,555640435,21190878,2021-08-11,6,88877547,"Zhu, Dongmin"
3,576046083,22184265,2022-07-12,0,90491685,"Zhou, Xu-Hong"
4,530646733,20173269,2020-05-06,2,77744450,"Zhou, Junqiang"
...,...,...,...,...,...,...
1412,537592943,20765210,2020-08-18,0,83404890,"AERTS, Jan"
1413,540288137,20205786,2020-11-04,1,83162946,"ADAMS, Peter"
1414,556618553,21194601,2021-09-02,0,83246691,"ABOUJDID, Eric"
1415,537734329,20195931,2020-09-14,0,83230223,"ABEYASEKERA, Tusitha"


### Aggregating citations 

Since we have a many-to-many relationship between persons and applications, we can see that some inventors have more than one granted patent resulting from our query. In order to see the ranking of influential inventors, we need to aggregate the `nb_citing_docdb_fam` field per unique name. For that we need to import a module from SQLAlchemy called `func`. This module contains several useful methods for querying data, such as the `sum()` method that we will use now. 


In [5]:
# Importing the func model
from sqlalchemy import func


# Define the query in ORM
q = db.query(
    TLS206_PERSON.person_id,
    TLS206_PERSON.person_name,  # inventor's name
    func.sum(TLS201_APPLN.nb_citing_docdb_fam).label('total_citations')  # sum of families citing patents by a given inventor
).join(
    TLS207_PERS_APPLN, TLS201_APPLN.appln_id == TLS207_PERS_APPLN.appln_id
).join(
    TLS206_PERSON, TLS207_PERS_APPLN.person_id == TLS206_PERSON.person_id
).filter(
    TLS201_APPLN.appln_filing_year >= 2020,
    TLS201_APPLN.appln_auth == 'EP',
    TLS201_APPLN.granted == 'Y',
    TLS207_PERS_APPLN.invt_seq_nr > 0  # filter to include only inventors
).group_by(
    TLS206_PERSON.person_id,
    TLS206_PERSON.person_name
).order_by(
    func.sum(TLS201_APPLN.nb_citing_docdb_fam).desc()  # order by total citations in descending order
)

# Creating a dataframe with the results
res = patstat.df(q)

res


,person_id,person_name,total_citations
0,81653189,"RÖDER, Roman",49
1,81612706,"PETER, Andreas",49
2,81632183,"XU, Xiaolan",25
3,81637282,"SIKORA, Bernhard",24
4,77711740,"Ray, Olive",12
...,...,...,...
1289,90491685,"Zhou, Xu-Hong",0
1290,90514874,"Yang, Qing-shan",0
1291,90486988,"Bai, Jiu-Lin",0
1292,90509313,"Ke, Ke",0
